In [1]:
from pathlib import Path
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "hyperparam_search"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_hparam_search_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
DATA_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/data
OUTPUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search
EXPERIMENT_ID: exp_hparam_search_20260321_211923


In [2]:
import itertools
import json
import pandas as pd

from configs.models import MODELS
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SAMPLE_PATH = DATA_DIR / "Muestra_csv.csv"

df_sample = pd.read_csv(SAMPLE_PATH)

print("Shape:", df_sample.shape)
print("Columnas:", list(df_sample.columns))
display(df_sample.head(3))

Shape: (20, 17)
Columnas: ['id', 'idFinal', 'grupo', 'motivo', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


,id,idFinal,grupo,motivo,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,Vivian,10,1,4,5.0,NaN,NaN,NaN,NaN,NaN,1
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Vivian,14,5,1,6.0,NaN,NaN,NaN,NaN,NaN,1
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Vivian,5,6,19,1.0,NaN,NaN,NaN,NaN,NaN,1


In [4]:
META_COLS = ["idFinal", "grupo", "motivo", "lex"]

required_cols = ["id", "Segmento", "Propuesta"]
missing = [c for c in required_cols if c not in df_sample.columns]

if missing:
    raise ValueError(f"Faltan columnas requeridas en la muestra: {missing}")

df_refine = df_sample.copy()

df_refine = df_refine.rename(
    columns={
        "id": "sample_id",
        "Segmento": "source_text",
        "Propuesta": "reference_text",
    }
)

keep_cols = ["sample_id", "source_text", "reference_text"] + [c for c in META_COLS if c in df_refine.columns]
df_refine = df_refine[keep_cols].copy()

print("Shape refinamiento:", df_refine.shape)
display(df_refine.head(5))

Shape refinamiento: (20, 7)


,sample_id,source_text,reference_text,idFinal,grupo,motivo,lex
0,2088,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,2976,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,3430,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1
3,3679,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,1
4,3145,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,1


In [5]:
FEW_SHOT_EXAMPLES = [
    {
        "source": """A continuación una tabla donde se demuestra como crecen los ahorros al pasar los años: Edad Inversión Intereses Saldo Inversión Intereses Saldo Laura Ganados Alberto Ganados 22 $800 $80 $880 23 $800 $168 $1.232 Valor al Jubilarse $311.092 Valor al Jubilarse $263.232 Menos las contribuciones iniciales $6.400 Menos las contribuciones iniciales $28.800 Ganancia Neta $304.692 Ganancia Neta $234.""",
        "target": """A continuación, presentamos una tabla donde se demuestra como crecen los ahorros al pasar los años. Las categorías en que se divide la tabla son edad, inversión, intereses ganados y saldo. La tabla contrasta las ganancias en intereses por año de Laura y Alberto con una proyección de edad de jubilación de sesenta y cinco años y según la cantidad de años que mantuvieron el ahorro."""
    },
    {
        "source": """El varias veces mencionado Yager, nos dice acertadamente sobre estos aspectos lo siguiente: La mayoría de ustedes están hambrientos de tener libertad financiera, están hastiados de tener batallas financieras porque son como un carrusel.""",
        "target": """Yager habla acertadamente sobre estos aspectos y dice que la mayoría ambiciona tener libertad financiera y está cansada de tener batallas financieras inútiles."""
    },
    {
        "source": """De acuerdo con esta medición, la proporción de pobres en la población mundial quienes viven con menos de un dólar por día descendió levemente entre 1987 y 1993, pues pasó del 30% al 29%.""",
        "target": """Según esta medición, la proporción de personas pobres en la población mundial descendió un poco entre mil novecientos ochenta y siete y mil novecientos noventa y tres. Es decir, el porcentaje de personas que vivían con menos de un dólar por día pasó del treinta al veintinueve por ciento."""
    },
    {
        "source": """- Rentabilidad
INTERÉS DE 2 PESOS SOBRE UNA INVERSIÓN DE 100 PESOS = RENTABILIDAD DE 2 % ((2 ÷ 100) × 100 = 2%) POR CADA 100 PESOS SE GANAN 2 PESOS
 Endeudamiento
 DEUDA DE 30 PESOS POR SOBRE EL CAPITAL DE 20 PESOS = ENDEUDAMIENTO DE 1,5 VECES EL CAPITAL (30 ÷ 20 = 1,5) POR CADA PESO DE CAPITAL EXISTE 1,5 PESOS DE DEUDA
 Liquidez
80 PESOS DE DINERO DISPONIBLE POR SOBRE DEUDAS DE 40 PESOS = LIQUIDEZ DE DOS VECES EL VALOR DE LA DEUDA (80 ÷ 40 = 2) POR CADA PESO DE DEUDA SE DISPONE DE 2 PESOS PARA PAGO""",
        "target": """EL INTERÉS DE DOS PESOS SOBRE UNA INVERSIÓN DE CIEN PESOS DA UNA RENTABILIDAD DE DOS POR CIENTO. ES DECIR, POR CADA CIEN PESOS GANAMOS DOS PESOS. LA DEUDA DE TREINTA PESOS SOBRE EL CAPITAL DE VEINTE PESOS RESULTA EN UNA DEUDA DE UNO PUNTO CINCO VECES EL CAPITAL. ES DECIR, POR CADA PESO DE CAPITAL HAY UNO PUNTO CINCO PESOS DE DEUDA. EN OCHENTA PESOS DE DINERO DISPONIBLE SOBRE DEUDAS DE CUARENTA PESOS DA UNA LIQUIDEZ DEL DOBLE DE LA DEUDA. POR CADA PESO ADEUDADO HAY DOS PESOS PARA PAGAR."""
    },
]

FEW_SHOT_EXAMPLE_IDS = [78, 1805, 3635, 5262]

print("Número de ejemplos few-shot:", len(FEW_SHOT_EXAMPLES))
print("IDs few-shot:", FEW_SHOT_EXAMPLE_IDS)

Número de ejemplos few-shot: 4
IDs few-shot: [78, 1805, 3635, 5262]


In [6]:
ACTIVE_MODELS = [
    "llama3",
    "mistral",
]

PROMPT_TYPE = "few-shot"
RULESET = "R4"

TEMPERATURES = [0.1, 0.2, 0.3]
TOP_PS = [0.85, 0.90]
REPETITION_PENALTIES = [1.05, 1.10, 1.15]
MAX_NEW_TOKENS = 256

PARAM_GRID = [
    {
        "temperature": t,
        "top_p": p,
        "repetition_penalty": r,
        "max_new_tokens": MAX_NEW_TOKENS,
    }
    for t, p, r in itertools.product(
        TEMPERATURES,
        TOP_PS,
        REPETITION_PENALTIES,
    )
]

print("Modelos:", ACTIVE_MODELS)
print("Prompt type fijo:", PROMPT_TYPE)
print("Ruleset fijo:", RULESET)
print("N configuraciones:", len(PARAM_GRID))
display(pd.DataFrame(PARAM_GRID).head(10))

Modelos: ['llama3', 'mistral']
Prompt type fijo: few-shot
Ruleset fijo: R4
N configuraciones: 18


,temperature,top_p,repetition_penalty,max_new_tokens
0,0.1,0.85,1.05,256
1,0.1,0.85,1.10,256
2,0.1,0.85,1.15,256
3,0.1,0.90,1.05,256
4,0.1,0.90,1.10,256
5,0.1,0.90,1.15,256
6,0.2,0.85,1.05,256
7,0.2,0.85,1.10,256
8,0.2,0.85,1.15,256
9,0.2,0.90,1.05,256


In [7]:
for model_key in ACTIVE_MODELS:
    if model_key not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {model_key}")

if RULESET not in RULESETS:
    raise ValueError(f"Ruleset no definido: {RULESET}")

if PROMPT_TYPE not in ["zero-shot", "few-shot"]:
    raise ValueError(f"Técnica no soportada: {PROMPT_TYPE}")

print("Configuración validada correctamente.")

Configuración validada correctamente.


In [8]:
runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("Runner inicializado:", runner.experiment_id)

Runner inicializado: exp_hparam_search_20260321_211923


In [9]:
test_row = df_refine.iloc[0]

test_record = runner.run_one(
    dataset_name="muestra_20_hparam_search",
    model_key=ACTIVE_MODELS[0],
    prompt_type=PROMPT_TYPE,
    ruleset=RULESET,
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["sample_id"]),
    fold_id=None,
    split_name="hparam_search",
    few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
    few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
    generation_config=PARAM_GRID[0],
)

test_record.to_dict()

{'experiment_id': 'exp_hparam_search_20260321_211923',
 'run_id': '68c62e8a-ac82-4aea-99fa-80ea841c938a',
 'timestamp': '2026-03-21T21:19:27.238321',
 'dataset_name': 'muestra_20_hparam_search',
 'fold_id': None,
 'split_name': 'hparam_search',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'few-shot',
 'ruleset': 'R4',
 'few_shot_example_ids': [78, 1805, 3635, 5262],
 'generation_config': {'temperature': 0.1,
  'top_p': 0.85,
  'repetition_penalty': 1.05,
  'max_new_tokens': 256},
 'sample_id': '2088',
 'source_text': 'La comunidad humana más antigua ha sido denominada horda primitiva.',
 'reference_text': 'La comunidad humana más antigua se llamó tribu.',
 'generated_text': 'La comunidad humana más antigua se llama horda primitiva.',
 'prompt_text': 'Reescribe en español cada texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\nGuía interna de simplificación: usa palab

In [10]:
all_records = []

total_runs = len(ACTIVE_MODELS) * len(PARAM_GRID) * len(df_refine)
run_counter = 0

for model_key in ACTIVE_MODELS:
    for generation_config in PARAM_GRID:
        for _, row in df_refine.iterrows():
            run_counter += 1
            print(
                f"[{run_counter}/{total_runs}] "
                f"model={model_key} | prompt={PROMPT_TYPE} | "
                f"ruleset={RULESET} | sample_id={row['sample_id']} | "
                f"temp={generation_config['temperature']} top_p={generation_config['top_p']} rep={generation_config['repetition_penalty']}"
            )

            record = runner.run_one(
                dataset_name="muestra_20_hparam_search",
                model_key=model_key,
                prompt_type=PROMPT_TYPE,
                ruleset=RULESET,
                source_text=row["source_text"],
                reference_text=row["reference_text"],
                sample_id=str(row["sample_id"]),
                fold_id=None,
                split_name="hparam_search",
                few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
                few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
                generation_config=generation_config,
            )

            record_dict = record.to_dict()

            for meta_col in ["idFinal", "grupo", "motivo", "lex"]:
                if meta_col in row.index:
                    record_dict[meta_col] = row[meta_col]

            all_records.append(record_dict)

print(f"Corridas completadas: {len(all_records)}")

[1/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=2088 | temp=0.1 top_p=0.85 rep=1.05
[2/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=2976 | temp=0.1 top_p=0.85 rep=1.05
[3/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=3430 | temp=0.1 top_p=0.85 rep=1.05
[4/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=3679 | temp=0.1 top_p=0.85 rep=1.05
[5/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=3145 | temp=0.1 top_p=0.85 rep=1.05
[6/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=507 | temp=0.1 top_p=0.85 rep=1.05
[7/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=1756 | temp=0.1 top_p=0.85 rep=1.05
[8/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=3093 | temp=0.1 top_p=0.85 rep=1.05
[9/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=3192 | temp=0.1 top_p=0.85 rep=1.05
[10/720] model=llama3 | prompt=few-shot | ruleset=R4 | sample_id=3525 | temp=0.1 top_p=0.85 rep=1.05


In [11]:
results_df = pd.DataFrame(all_records)

print("Shape results_df:", results_df.shape)
display(results_df.head(3))

if "status" in results_df.columns:
    display(results_df["status"].value_counts(dropna=False))

Shape results_df: (720, 26)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,generated_text,prompt_text,inference_seconds,status,error_message,metrics,idFinal,grupo,motivo,lex
0,exp_hparam_search_20260321_211923,963f6e0c-1944-4893-8323-6ee1af3e53b2,2026-03-21T21:19:35.100956,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,La comunidad humana más antigua se llama horda...,Reescribe en español cada texto con lenguaje m...,23.4045,success,None,{},1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_hparam_search_20260321_211923,123d8140-bcef-4997-ac5e-652622a0160c,2026-03-21T21:19:58.592780,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Prueba a guardar cada día $1 y también las mon...,Reescribe en español cada texto con lenguaje m...,9.2751,success,None,{},881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_hparam_search_20260321_211923,46d6d637-47c4-44ad-9ece-e1dc4a5e9040,2026-03-21T21:20:07.948050,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Uno de los problemas más grandes en los negoci...,Reescribe en español cada texto con lenguaje m...,12.2155,success,None,{},2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


status
success    720
Name: count, dtype: int64

In [12]:
raw_results_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"
results_df.to_csv(raw_results_path, index=False, encoding="utf-8-sig")

print(raw_results_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search/exp_hparam_search_20260321_211923_raw_results.csv


In [13]:
successful_df = results_df[results_df["status"] == "success"].copy()

print("Corridas exitosas:", len(successful_df))
print("Corridas con error:", len(results_df) - len(successful_df))
display(successful_df.head(3))

Corridas exitosas: 720
Corridas con error: 0


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,generated_text,prompt_text,inference_seconds,status,error_message,metrics,idFinal,grupo,motivo,lex
0,exp_hparam_search_20260321_211923,963f6e0c-1944-4893-8323-6ee1af3e53b2,2026-03-21T21:19:35.100956,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,La comunidad humana más antigua se llama horda...,Reescribe en español cada texto con lenguaje m...,23.4045,success,None,{},1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_hparam_search_20260321_211923,123d8140-bcef-4997-ac5e-652622a0160c,2026-03-21T21:19:58.592780,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Prueba a guardar cada día $1 y también las mon...,Reescribe en español cada texto con lenguaje m...,9.2751,success,None,{},881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_hparam_search_20260321_211923,46d6d637-47c4-44ad-9ece-e1dc4a5e9040,2026-03-21T21:20:07.948050,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Uno de los problemas más grandes en los negoci...,Reescribe en español cada texto con lenguaje m...,12.2155,success,None,{},2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


In [14]:
evaluated_df = evaluate_dataframe(
    successful_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

Shape evaluated_df: (720, 51)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_hparam_search_20260321_211923,963f6e0c-1944-4893-8323-6ee1af3e53b2,2026-03-21T21:19:35.100956,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.222222,0.300000,52.468333,34.855000,17.613333,0.736842,0.705882,0.736842,0.912748,None
1,exp_hparam_search_20260321_211923,123d8140-bcef-4997-ac5e-652622a0160c,2026-03-21T21:19:58.592780,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.500000,0.562500,99.385000,105.172500,-5.787500,0.200000,0.071429,0.200000,0.758098,None
2,exp_hparam_search_20260321_211923,46d6d637-47c4-44ad-9ece-e1dc4a5e9040,2026-03-21T21:20:07.948050,muestra_20_hparam_search,None,hparam_search,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.294118,0.294118,79.893824,76.229118,3.664706,0.266667,0.071429,0.133333,0.839404,None


In [15]:
param_cols = evaluated_df["generation_config"].apply(pd.Series)
param_cols = param_cols.rename(columns=lambda c: f"gen_{c}")

evaluated_df = pd.concat([evaluated_df, param_cols], axis=1)

display(
    evaluated_df[
        [
            "model_key",
            "prompt_type",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
        ]
    ].head(5)
)

,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens
0,llama3,few-shot,R4,0.1,0.85,1.05,256.0
1,llama3,few-shot,R4,0.1,0.85,1.05,256.0
2,llama3,few-shot,R4,0.1,0.85,1.05,256.0
3,llama3,few-shot,R4,0.1,0.85,1.05,256.0
4,llama3,few-shot,R4,0.1,0.85,1.05,256.0


In [16]:
evaluated_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"
evaluated_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print(evaluated_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search/exp_hparam_search_20260321_211923_evaluated.csv


In [17]:
# Celda 16
summary_by_full_config = summarize_metrics(
    evaluated_df,
    group_cols=[
        "model_key",
        "prompt_type",
        "ruleset",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ],
)

display(
    summary_by_full_config.sort_values(
        by=["sari", "bertscore_f1", "rougeL_f"],
        ascending=False
    ).head(30)
)

summary_by_full_config_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_full_config.csv"
summary_by_full_config.to_csv(summary_by_full_config_path, index=False, encoding="utf-8-sig")

print(summary_by_full_config_path)

,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
6,llama3,few-shot,R4,0.2,0.85,1.05,256.0,43.736481,19.991764,85.598532,...,0.0,0.276485,0.455337,0.488855,0.327324,0.427845,71.968281,51.983615,19.984666,0.800956
10,llama3,few-shot,R4,0.2,0.90,1.10,256.0,43.609996,17.464022,83.856426,...,0.0,0.368046,0.514652,0.466428,0.303590,0.413543,69.832007,51.983615,17.848392,0.802792
17,llama3,few-shot,R4,0.3,0.90,1.15,256.0,42.994219,16.406612,80.353769,...,0.0,0.426762,0.535183,0.437145,0.272437,0.382567,66.399268,51.983615,14.415653,0.796446
3,llama3,few-shot,R4,0.1,0.90,1.05,256.0,42.760489,18.315438,86.601227,...,0.0,0.274244,0.439577,0.513089,0.326600,0.445096,72.288790,51.983615,20.305175,0.810652
11,llama3,few-shot,R4,0.2,0.90,1.15,256.0,42.536652,14.394945,81.439478,...,0.0,0.436527,0.558060,0.445273,0.270763,0.383981,67.528034,51.983615,15.544420,0.799421
15,llama3,few-shot,R4,0.3,0.90,1.05,256.0,42.496364,18.077351,84.847113,...,0.0,0.305044,0.492456,0.474861,0.311164,0.423331,70.756884,51.983615,18.773269,0.801370
9,llama3,few-shot,R4,0.2,0.90,1.05,256.0,42.434082,17.374716,83.902856,...,0.0,0.319685,0.468641,0.467627,0.293690,0.423032,69.943217,51.983615,17.959602,0.802213
7,llama3,few-shot,R4,0.2,0.85,1.10,256.0,41.970640,12.659302,82.677631,...,0.0,0.378859,0.507074,0.449353,0.256547,0.361311,69.180527,51.983615,17.196913,0.796781
5,llama3,few-shot,R4,0.1,0.90,1.15,256.0,41.411878,14.492050,82.817489,...,0.0,0.420050,0.557960,0.434126,0.255459,0.363302,68.653420,51.983615,16.669805,0.794803
2,llama3,few-shot,R4,0.1,0.85,1.15,256.0,41.355371,14.723653,83.386716,...,0.0,0.406207,0.546867,0.432748,0.263293,0.368498,69.329620,51.983615,17.346005,0.791155


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search/exp_hparam_search_20260321_211923_summary_by_full_config.csv


In [18]:
leaderboard = summary_by_full_config.copy()

leaderboard = leaderboard.sort_values(
    by=["model_key", "sari", "bertscore_f1"],
    ascending=[True, False, False]
).reset_index(drop=True)

display(leaderboard.head(20))

,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,few-shot,R4,0.2,0.85,1.05,256.0,43.736481,19.991764,85.598532,...,0.0,0.276485,0.455337,0.488855,0.327324,0.427845,71.968281,51.983615,19.984666,0.800956
1,llama3,few-shot,R4,0.2,0.90,1.10,256.0,43.609996,17.464022,83.856426,...,0.0,0.368046,0.514652,0.466428,0.303590,0.413543,69.832007,51.983615,17.848392,0.802792
2,llama3,few-shot,R4,0.3,0.90,1.15,256.0,42.994219,16.406612,80.353769,...,0.0,0.426762,0.535183,0.437145,0.272437,0.382567,66.399268,51.983615,14.415653,0.796446
3,llama3,few-shot,R4,0.1,0.90,1.05,256.0,42.760489,18.315438,86.601227,...,0.0,0.274244,0.439577,0.513089,0.326600,0.445096,72.288790,51.983615,20.305175,0.810652
4,llama3,few-shot,R4,0.2,0.90,1.15,256.0,42.536652,14.394945,81.439478,...,0.0,0.436527,0.558060,0.445273,0.270763,0.383981,67.528034,51.983615,15.544420,0.799421
5,llama3,few-shot,R4,0.3,0.90,1.05,256.0,42.496364,18.077351,84.847113,...,0.0,0.305044,0.492456,0.474861,0.311164,0.423331,70.756884,51.983615,18.773269,0.801370
6,llama3,few-shot,R4,0.2,0.90,1.05,256.0,42.434082,17.374716,83.902856,...,0.0,0.319685,0.468641,0.467627,0.293690,0.423032,69.943217,51.983615,17.959602,0.802213
7,llama3,few-shot,R4,0.2,0.85,1.10,256.0,41.970640,12.659302,82.677631,...,0.0,0.378859,0.507074,0.449353,0.256547,0.361311,69.180527,51.983615,17.196913,0.796781
8,llama3,few-shot,R4,0.1,0.90,1.15,256.0,41.411878,14.492050,82.817489,...,0.0,0.420050,0.557960,0.434126,0.255459,0.363302,68.653420,51.983615,16.669805,0.794803
9,llama3,few-shot,R4,0.1,0.85,1.15,256.0,41.355371,14.723653,83.386716,...,0.0,0.406207,0.546867,0.432748,0.263293,0.368498,69.329620,51.983615,17.346005,0.791155


In [19]:
candidate_df = leaderboard.copy()

candidate_df = candidate_df[
    (candidate_df["exact_copy"] < 0.50) &
    (candidate_df["compression_ratio_eval"] > 0.60) &
    (candidate_df["compression_ratio_eval"] < 1.60)
].copy()

print("Configuraciones tras descarte:", len(candidate_df))
display(candidate_df.head(20))

Configuraciones tras descarte: 31


,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,few-shot,R4,0.2,0.85,1.05,256.0,43.736481,19.991764,85.598532,...,0.0,0.276485,0.455337,0.488855,0.327324,0.427845,71.968281,51.983615,19.984666,0.800956
1,llama3,few-shot,R4,0.2,0.90,1.10,256.0,43.609996,17.464022,83.856426,...,0.0,0.368046,0.514652,0.466428,0.303590,0.413543,69.832007,51.983615,17.848392,0.802792
2,llama3,few-shot,R4,0.3,0.90,1.15,256.0,42.994219,16.406612,80.353769,...,0.0,0.426762,0.535183,0.437145,0.272437,0.382567,66.399268,51.983615,14.415653,0.796446
3,llama3,few-shot,R4,0.1,0.90,1.05,256.0,42.760489,18.315438,86.601227,...,0.0,0.274244,0.439577,0.513089,0.326600,0.445096,72.288790,51.983615,20.305175,0.810652
4,llama3,few-shot,R4,0.2,0.90,1.15,256.0,42.536652,14.394945,81.439478,...,0.0,0.436527,0.558060,0.445273,0.270763,0.383981,67.528034,51.983615,15.544420,0.799421
5,llama3,few-shot,R4,0.3,0.90,1.05,256.0,42.496364,18.077351,84.847113,...,0.0,0.305044,0.492456,0.474861,0.311164,0.423331,70.756884,51.983615,18.773269,0.801370
6,llama3,few-shot,R4,0.2,0.90,1.05,256.0,42.434082,17.374716,83.902856,...,0.0,0.319685,0.468641,0.467627,0.293690,0.423032,69.943217,51.983615,17.959602,0.802213
7,llama3,few-shot,R4,0.2,0.85,1.10,256.0,41.970640,12.659302,82.677631,...,0.0,0.378859,0.507074,0.449353,0.256547,0.361311,69.180527,51.983615,17.196913,0.796781
8,llama3,few-shot,R4,0.1,0.90,1.15,256.0,41.411878,14.492050,82.817489,...,0.0,0.420050,0.557960,0.434126,0.255459,0.363302,68.653420,51.983615,16.669805,0.794803
9,llama3,few-shot,R4,0.1,0.85,1.15,256.0,41.355371,14.723653,83.386716,...,0.0,0.406207,0.546867,0.432748,0.263293,0.368498,69.329620,51.983615,17.346005,0.791155


In [21]:
top3_by_model = (
    candidate_df
    .sort_values(by=["model_key", "sari", "bertscore_f1"], ascending=[True, False, False])
    .groupby("model_key", as_index=False, group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

display(top3_by_model)

top3_by_model_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_top3_by_model.csv"
top3_by_model.to_csv(top3_by_model_path, index=False, encoding="utf-8-sig")

print(top3_by_model_path)

,model_key,prompt_type,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,few-shot,R4,0.2,0.85,1.05,256.0,43.736481,19.991764,85.598532,...,0.0,0.276485,0.455337,0.488855,0.327324,0.427845,71.968281,51.983615,19.984666,0.800956
1,llama3,few-shot,R4,0.2,0.90,1.10,256.0,43.609996,17.464022,83.856426,...,0.0,0.368046,0.514652,0.466428,0.303590,0.413543,69.832007,51.983615,17.848392,0.802792
2,llama3,few-shot,R4,0.3,0.90,1.15,256.0,42.994219,16.406612,80.353769,...,0.0,0.426762,0.535183,0.437145,0.272437,0.382567,66.399268,51.983615,14.415653,0.796446
3,mistral,few-shot,R4,0.3,0.90,1.10,256.0,40.185657,11.504663,81.024452,...,0.0,0.504514,0.344750,0.441418,0.239016,0.379856,53.604366,51.983615,1.620752,0.777665
4,mistral,few-shot,R4,0.1,0.85,1.10,256.0,39.771024,11.544617,87.182222,...,0.0,0.480489,0.333917,0.438303,0.241987,0.371807,58.363346,51.983615,6.379731,0.783501
5,mistral,few-shot,R4,0.2,0.85,1.10,256.0,39.381730,11.818992,81.935027,...,0.0,0.505045,0.352222,0.443705,0.251003,0.374098,54.156220,51.983615,2.172606,0.782989


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search/exp_hparam_search_20260321_211923_top3_by_model.csv


In [22]:
finalist_configs = top3_by_model[
    [
        "model_key",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ]
].drop_duplicates()

finalist_cases = evaluated_df.merge(
    finalist_configs,
    on=[
        "model_key",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ],
    how="inner"
)

qual_cols = [
    "sample_id",
    "idFinal",
    "grupo",
    "motivo",
    "model_key",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "gen_max_new_tokens",
    "source_text",
    "reference_text",
    "generated_text",
    "sari",
    "bertscore_f1",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
]

qual_cols = [c for c in qual_cols if c in finalist_cases.columns]

finalist_qual_review = finalist_cases[qual_cols].copy()

display(finalist_qual_review.head(20))

finalist_qual_review_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_finalist_qual_review.csv"
finalist_qual_review.to_csv(finalist_qual_review_path, index=False, encoding="utf-8-sig")

print(finalist_qual_review_path)

,sample_id,idFinal,grupo,motivo,model_key,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,0.2,0.85,1.05,256.0,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llama horda...,67.760943,0.912748,0.736842,0.900000,0
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,0.2,0.85,1.05,256.0,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Prueba a guardar $1 al día y también las moned...,35.479021,0.780270,0.181818,1.062500,0
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,llama3,0.2,0.85,1.05,256.0,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,La asignación de precios a los productos es un...,66.089081,0.930297,0.666667,0.823529,0
3,3679,829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,llama3,0.2,0.85,1.05,256.0,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",El IVA del kilo de pan es de $190.,32.491125,0.751794,0.344828,0.600000,0
4,3145,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,llama3,0.2,0.85,1.05,256.0,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,"Por cada $100 vendidos, la empresa gasta $2.20...",29.027778,0.773997,0.230769,1.000000,0
5,507,562_LibroUAC_Sincopyright.pdf,B_medianos,Porcentajes y rentabilidad.,llama3,0.2,0.85,1.05,256.0,"En general, se dice que la rentabilidad normal...","Generalmente, la rentabilidad normal de una in...",La rentabilidad normal de una inversión en paí...,60.983625,0.921257,0.800000,0.708333,0
6,1756,5100_LibroBAC.pdf,B_medianos,Redacción abstracta/institucional.,llama3,0.2,0.85,1.05,256.0,En virtud de estas y otras reflexiones fue que...,Debido a estas y otras reflexiones se diseñó e...,Este dinero es para ti.,28.576764,0.633102,0.071429,0.200000,0
7,3093,875_LibroUide_Sincopyright.pdf,B_medianos,Definición contable.,llama3,0.2,0.85,1.05,256.0,Las cuentas de activo reflejan aquello que pos...,Las cuentas de activo evidencian las pertenenc...,Las cuentas de activo muestran lo que una orga...,59.611993,0.820760,0.352941,1.000000,0
8,3192,1202_LibroUide_Sincopyright.pdf,B_medianos,"Relación de pasivo/activo, útil para precisión.",llama3,0.2,0.85,1.05,256.0,"Por cada dólar de pasivo corriente, la compañí...",La empresa tiene noventa y tres centavos de dó...,"Por cada dólar de deuda corriente, la empresa ...",43.725828,0.852730,0.523810,0.842105,0
9,3525,230_LibroUAC_Sincopyright.pdf,B_medianos,Dato económico con cifra.,llama3,0.2,0.85,1.05,256.0,Posee la renta per cápita más alta de Latinoam...,Tiene la renta per cápita más alta de Latinoam...,La renta per cápita más alta en Latinoamérica ...,46.623287,0.777711,0.500000,0.904762,0


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search/exp_hparam_search_20260321_211923_finalist_qual_review.csv


In [24]:
bitacora = {
    "experiment_id": EXPERIMENT_ID,
    "dataset_name": "muestra_20_hparam_search",
    "n_samples": int(len(df_refine)),
    "models_tested": ACTIVE_MODELS,
    "prompt_type_fixed": PROMPT_TYPE,
    "ruleset_fixed": RULESET,
    "temperatures": TEMPERATURES,
    "top_ps": TOP_PS,
    "repetition_penalties": REPETITION_PENALTIES,
    "max_new_tokens": MAX_NEW_TOKENS,
    "n_param_configs": int(len(PARAM_GRID)),
    "expected_total_runs": int(len(ACTIVE_MODELS) * len(PARAM_GRID) * len(df_refine)),
    "successful_runs": int(len(successful_df)),
    "leader_metrics": ["sari", "bertscore_f1"],
    "support_metrics": ["rougeL_f", "compression_ratio_eval", "exact_copy"],
    "discard_rules": {
        "exact_copy_max": 0.50,
        "compression_ratio_eval_min": 0.60,
        "compression_ratio_eval_max": 1.60,
    },
    "top3_by_model": top3_by_model.to_dict(orient="records"),
}

bitacora_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_bitacora_experimento.json"

with open(bitacora_path, "w", encoding="utf-8") as f:
    json.dump(bitacora, f, ensure_ascii=False, indent=2)

print(bitacora_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/hyperparam_search/exp_hparam_search_20260321_211923_bitacora_experimento.json


## Resumen de resultados

En este notebook se hizo una exploracion mas amplia de hiperparametros para los modelos `llama3` y `mistral`.  
A diferencia del notebook anterior, aqui ya no se probaron solo dos configuraciones, sino una malla mas grande variando `temperature`, `top_p` y `repetition_penalty`, manteniendo fija la tecnica `few-shot` y el ruleset `R4`.

En total se evaluaron 18 configuraciones por modelo sobre la muestra de 20 textos, dando un total de 720 corridas. Todas las corridas terminaron bien, asi que esta fase ya sirve como una exploracion formal de hiperparametros dentro del proyecto.

Para hacer la seleccion no se uso una sola metrica, porque este problema es multiobjetivo. En esta etapa se tomaron como metricas lideres **SARI** y **BERTScore F1**, mientras que otras metricas como `rougeL_f`, `compression_ratio_eval` y `exact_copy` se usaron como apoyo para descartar configuraciones menos utiles.

De manera general, `llama3` fue el modelo que mostro el mejor comportamiento en esta fase, ya que sus configuraciones ocuparon los primeros lugares del ranking. Aun asi no se eligio un unico ganador final, sino que se conservaron las **3 mejores configuraciones por modelo**, con la idea de pasar despues a una segunda etapa donde se comparen otras variables, por ejemplo los distintos conjuntos de reglas.

En otras palabras, este notebook no cierra todavia la seleccion final del sistema, pero si reduce bastante el universo de opciones y deja finalistas bien sustentados para seguir con los siguientes experimentos.